<a href="https://colab.research.google.com/github/Ram-Vidhu/Job_searcher_and_Resume_Enhancer/blob/users%2Fvidhya%2Fgen_ai/Notebooks/GenAI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Spark Setup

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [ ]:
!ls

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

## Data preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df1 = spark.read.format('csv').option("header", True).load('/content/drive/MyDrive/Datasets/job_skills.csv',)

In [ ]:
df2 = spark.read.format('csv').option("header", True).load('/content/drive/MyDrive/Datasets/job_summary.csv')

In [ ]:
df3 = df1.join(df2,on='job_link',how='inner')

In [ ]:
df4 = spark.read.format('csv').option("header", True).load('/content/drive/MyDrive/Datasets/linkedin_job_postings.csv')

In [ ]:
df5 = df3.join(df4,on='job_link',how='inner')
df5.drop('last_processed_time','got_summary','got_ner','is_being_worked')

## EDA

In [ ]:
from pyspark.sql import functions as f

# take null counts
null_counts = df5.select([
    sum(f.col(c).isNull().cast("int")).alias(c)
    for c in df5.columns
])

null_counts.show()


In [ ]:
# dropping null values
df5 = df5.na.drop()

## Storing and querying vectordb

In [ ]:
!pip install chromadb

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
from pyspark.sql import functions as F

df5 = df5.withColumn(
    "job_text",
    F.concat_ws(
        " ",   # separator
        F.coalesce(F.col("job_title"), F.lit("")),
        F.coalesce(F.col("job_summary"), F.lit("")),
        F.coalesce(F.col("job_skills"), F.lit("")),
        F.coalesce(F.col("job_level"), F.lit(""))
    )
)

In [ ]:
# Init Chroma client (persistent storage)
client = chromadb.PersistentClient(path="chroma_db")

# Create or get collection
collection = client.get_or_create_collection(
    name="jobs",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

In [ ]:
# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
def clean_metadata(row_dict):
    clean = {}
    for k, v in row_dict.items():
        if k == "job_text":   # don't include job_text in metadata
            continue
        if v is None:
            clean[k] = ""   # default to empty string
        elif isinstance(v, (bool, int, float, str)):
            clean[k] = v
        else:
            clean[k] = str(v)   # fallback: convert to string
    return clean

In [ ]:
batch_size = 500
rows_iter = df5.toLocalIterator()

batch = []
for row in rows_iter:
    batch.append(row.asDict())

    if len(batch) >= batch_size:
        texts = [r["job_text"] for r in batch]
        embeddings = model.encode(texts)

        ids = [str(i) for i in range(len(batch))]
        metadata = [clean_metadata(r) for r in batch]

        collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadata
        )
        batch = []

In [ ]:
def search_jobs_chroma(resume_text, top_k=5, filters=None):
    embedding = model.encode([resume_text])[0]

    results = collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=top_k,
        where=filters  # e.g., {"job_location": "Berlin", "job_type": "Full-time"}
    )

    jobs = []
    for i in range(len(results["ids"][0])):
        jobs.append({
            "similarity_score": results["distances"][0][i],
            **results["metadatas"][0][i]
        ))
        return pd.DataFrame(jobs)

In [ ]:
results = search_jobs_chroma("data scientist [python, sql, machine learning, pyspark, Azure] mid level", top_k=5)

In [ ]:
results